# Phase 3-6: Publication Visualizations

Generate figures for the final paper.

In [ ]:
import sys
import os
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc

# nicer plots
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 11
sns.set_style("whitegrid")

# colab
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = "/content/drive/MyDrive/Colab Notebooks/data"
else:
    BASE = "/Users/tyreecruse/Desktop/CS230/Project/Data"

FEATURES_DIR = f"{BASE}/analysis/yolo_features"
RESULTS_DIR = f"{BASE}/analysis/results"
FIGURES_DIR = f"{BASE}/analysis/figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

print(f"Saving figures to: {FIGURES_DIR}")

In [ ]:
# load clean features
with open(f"{FEATURES_DIR}/clean_yolo_features.pkl", 'rb') as f:
    data = pickle.load(f)
cleanFeats = data['features']
print(f"clean: {cleanFeats.shape}")

centroid = cleanFeats.mean(axis=0)
centroid = centroid / np.linalg.norm(centroid)
cleanDist = 1 - (cleanFeats @ centroid)

In [ ]:
# UMAP figure
print("\nGenerating UMAP...")

try:
    import umap
except:
    !pip install umap-learn -q
    import umap

# sample 500 from clean
np.random.seed(42)
idx = np.random.choice(len(cleanFeats), 500, replace=False)
all_feats = [cleanFeats[idx]]
all_labels = ['Clean'] * 500
all_colors = ['blue'] * 500

# add some attacks
attacks_to_show = [
    ('fgsm_045', 'FGSM ε=0.045', 'red'),
    ('fgsm_105', 'FGSM ε=0.105', 'darkred'),
    ('gaussian_050', 'Gaussian σ=0.05', 'lightgreen'),
    ('gaussian_250', 'Gaussian σ=0.25', 'darkgreen'),
    ('patches', 'Patches', 'purple'),
]

for atk, label, color in attacks_to_show:
    path = f"{FEATURES_DIR}/{atk}_yolo_features.pkl"
    if os.path.exists(path):
        with open(path, 'rb') as f:
            data = pickle.load(f)
        feats = data['features']
        idx = np.random.choice(len(feats), min(500, len(feats)), replace=False)
        all_feats.append(feats[idx])
        all_labels.extend([label] * len(idx))
        all_colors.extend([color] * len(idx))
        print(f"  {atk}: {len(idx)} samples")

X = np.vstack(all_feats)
print(f"Total: {len(X)}")

# run umap
print("Running UMAP...")
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
emb = reducer.fit_transform(X)
print("Done")

In [ ]:
# plot
plt.figure(figsize=(10, 8))

# get unique labels in order
unique_labels = ['Clean'] + [a[1] for a in attacks_to_show]
color_map = {'Clean': 'blue'}
for atk, label, color in attacks_to_show:
    color_map[label] = color

for label in unique_labels:
    mask = [l == label for l in all_labels]
    if any(mask):
        plt.scatter(emb[mask, 0], emb[mask, 1], c=color_map[label], 
                   label=label, alpha=0.6, s=20)

plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.title('YOLOv8 Feature Space')
plt.legend()
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/fig1_umap.png", dpi=300, bbox_inches='tight')
plt.savefig(f"{FIGURES_DIR}/fig1_umap.pdf", bbox_inches='tight')
plt.show()
print("saved fig1_umap")

In [ ]:
# AUC vs strength figure
print("\nComputing AUCs...")

fgsm_attacks = ['fgsm_030', 'fgsm_045', 'fgsm_060', 'fgsm_075', 'fgsm_090', 'fgsm_105']
gauss_attacks = ['gaussian_010', 'gaussian_050', 'gaussian_150', 'gaussian_200', 'gaussian_250']

fgsm_results = []
for atk in fgsm_attacks:
    path = f"{FEATURES_DIR}/{atk}_yolo_features.pkl"
    if os.path.exists(path):
        with open(path, 'rb') as f:
            data = pickle.load(f)
        advDist = 1 - (data['features'] @ centroid)
        allDist = np.concatenate([cleanDist, advDist])
        labels = np.concatenate([np.zeros(len(cleanDist)), np.ones(len(advDist))])
        fpr, tpr, _ = roc_curve(labels, allDist)
        strength = int(atk.split('_')[1]) / 1000
        fgsm_results.append((strength, auc(fpr, tpr)))

gauss_results = []
for atk in gauss_attacks:
    path = f"{FEATURES_DIR}/{atk}_yolo_features.pkl"
    if os.path.exists(path):
        with open(path, 'rb') as f:
            data = pickle.load(f)
        advDist = 1 - (data['features'] @ centroid)
        allDist = np.concatenate([cleanDist, advDist])
        labels = np.concatenate([np.zeros(len(cleanDist)), np.ones(len(advDist))])
        fpr, tpr, _ = roc_curve(labels, allDist)
        strength = int(atk.split('_')[1]) / 1000
        gauss_results.append((strength, auc(fpr, tpr)))

print(f"FGSM: {len(fgsm_results)} points")
print(f"Gaussian: {len(gauss_results)} points")

In [ ]:
# plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# fgsm
fgsm_results.sort()
x = [r[0] for r in fgsm_results]
y = [r[1] for r in fgsm_results]
axes[0].plot(x, y, 'ro-', lw=2, markersize=10)
axes[0].axhline(0.9, color='green', ls='--', alpha=0.5)
axes[0].axhline(0.5, color='gray', ls='--', alpha=0.5)
axes[0].set_xlabel('FGSM ε')
axes[0].set_ylabel('ROC-AUC')
axes[0].set_title('Detection vs FGSM Strength')
axes[0].set_ylim([0.5, 1.0])
axes[0].grid(alpha=0.3)

# gaussian
gauss_results.sort()
x = [r[0] for r in gauss_results]
y = [r[1] for r in gauss_results]
axes[1].plot(x, y, 'go-', lw=2, markersize=10)
axes[1].axhline(0.9, color='green', ls='--', alpha=0.5)
axes[1].axhline(0.5, color='gray', ls='--', alpha=0.5)
axes[1].set_xlabel('Gaussian σ')
axes[1].set_ylabel('ROC-AUC')
axes[1].set_title('Detection vs Gaussian Strength')
axes[1].set_ylim([0.4, 1.0])
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/fig2_auc_vs_strength.png", dpi=300, bbox_inches='tight')
plt.savefig(f"{FIGURES_DIR}/fig2_auc_vs_strength.pdf", bbox_inches='tight')
plt.show()
print("saved fig2_auc_vs_strength")

In [ ]:
# distance histograms
print("\nGenerating histograms...")

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.flatten()

attacks_for_hist = [
    ('fgsm_045', 'FGSM ε=0.045', 'red'),
    ('fgsm_105', 'FGSM ε=0.105', 'darkred'),
    ('gaussian_050', 'Gaussian σ=0.05', 'lightgreen'),
    ('gaussian_250', 'Gaussian σ=0.25', 'darkgreen'),
    ('patches', 'Patches', 'purple'),
]

for i, (atk, label, color) in enumerate(attacks_for_hist):
    ax = axes[i]
    path = f"{FEATURES_DIR}/{atk}_yolo_features.pkl"
    
    if os.path.exists(path):
        with open(path, 'rb') as f:
            data = pickle.load(f)
        advDist = 1 - (data['features'] @ centroid)
        
        ax.hist(cleanDist, bins=50, alpha=0.6, density=True, label='Clean', color='blue')
        ax.hist(advDist, bins=50, alpha=0.6, density=True, label=label, color=color)
        
        # 3-sigma line
        thresh = cleanDist.mean() + 3*cleanDist.std()
        ax.axvline(thresh, color='black', ls='--', lw=2, label='3σ')
        
        ax.set_xlabel('Distance')
        ax.set_ylabel('Density')
        ax.set_title(label)
        ax.legend(fontsize=9)
    else:
        ax.text(0.5, 0.5, 'Not found', ha='center', va='center', transform=ax.transAxes)

axes[5].set_visible(False)  # hide extra

plt.suptitle('Distance Distributions', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/fig3_histograms.png", dpi=300, bbox_inches='tight')
plt.savefig(f"{FIGURES_DIR}/fig3_histograms.pdf", bbox_inches='tight')
plt.show()
print("saved fig3_histograms")

In [ ]:
# LaTeX table
print("\n" + "="*60)
print("LATEX TABLE")
print("="*60)

all_attacks = fgsm_attacks + gauss_attacks + ['patches']
table_rows = []

for atk in all_attacks:
    path = f"{FEATURES_DIR}/{atk}_yolo_features.pkl"
    if not os.path.exists(path):
        continue
    
    with open(path, 'rb') as f:
        data = pickle.load(f)
    advDist = 1 - (data['features'] @ centroid)
    
    # auc
    allDist = np.concatenate([cleanDist, advDist])
    labels = np.concatenate([np.zeros(len(cleanDist)), np.ones(len(advDist))])
    fpr, tpr, _ = roc_curve(labels, allDist)
    roc_auc = auc(fpr, tpr)
    
    # 3-sigma
    thresh = cleanDist.mean() + 3*cleanDist.std()
    recall_3s = (advDist > thresh).mean()
    fpr_3s = (cleanDist > thresh).mean()
    
    # parse type
    if 'fgsm' in atk:
        atype = 'FGSM'
        strength = int(atk.split('_')[1]) / 1000
    elif 'gaussian' in atk:
        atype = 'Gaussian'
        strength = int(atk.split('_')[1]) / 1000
    else:
        atype = 'Patch'
        strength = None
    
    table_rows.append((atype, strength, roc_auc, recall_3s, fpr_3s))

# print latex
print(r"\begin{tabular}{lcccc}")
print(r"\hline")
print(r"Attack & Strength & AUC & 3$\sigma$ Recall & 3$\sigma$ FPR \\")
print(r"\hline")

table_rows.sort(key=lambda x: (x[0], x[1] or 0))
for atype, strength, roc_auc, recall, fpr in table_rows:
    s = f"{strength:.3f}" if strength else "-"
    print(f"{atype} & {s} & {roc_auc:.3f} & {recall:.3f} & {fpr:.4f} \\\\")

print(r"\hline")
print(r"\end{tabular}")

In [ ]:
# save csv too
df = pd.DataFrame(table_rows, columns=['type', 'strength', 'auc', 'recall_3s', 'fpr_3s'])
df.to_csv(f"{RESULTS_DIR}/detection_summary.csv", index=False)
print(f"\nsaved detection_summary.csv")

# list files
print("\nGenerated files:")
for f in sorted(os.listdir(FIGURES_DIR)):
    if f.startswith('fig'):
        print(f"  {f}")